# 🛢️ Oil Storage Tank Detection — Exploratory Data Analysis

This notebook analyses the Airbus Oil Storage Detection dataset before training.
Run after `python scripts/prepare_data.py --download`.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from PIL import Image

plt.style.use('dark_background')

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
TILE_W, TILE_H = 640, 640


## 1. Raw Dataset Overview


In [ ]:
import sys; sys.path.insert(0, '../scripts')
from prepare_data import parse_annotations

annotations = parse_annotations(RAW_DIR)
img_paths = sorted((RAW_DIR / 'images').glob('*.jpg'))

print(f'Images         : {len(img_paths)}')
print(f'Annotated      : {len(annotations)}')
total = sum(len(v) for v in annotations.values())
print(f'Total tanks    : {total}')
print(f'Avg per image  : {total/max(len(annotations),1):.1f}')


## 2. Bounding Box Size Distribution


In [ ]:
widths, heights, areas = [], [], []
for boxes in annotations.values():
    for b in boxes:
        w = b[2]-b[0]; h = b[3]-b[1]
        if w > 0 and h > 0:
            widths.append(w); heights.append(h); areas.append(w*h)

diameters = [(w+h)/2 for w,h in zip(widths,heights)]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(widths,    bins=50, color='#00d4ff', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Width (px)'); axes[0].set_title('Box Width Distribution')

axes[1].hist(heights,   bins=50, color='#00e676', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Height (px)'); axes[1].set_title('Box Height Distribution')

axes[2].hist(diameters, bins=50, color='#ffd700', edgecolor='black', alpha=0.8)
axes[2].axvline(np.mean(diameters), color='red', lw=2, linestyle='--',
                label=f'Mean={np.mean(diameters):.0f}px')
axes[2].set_xlabel('Diameter px (mean of w,h)'); axes[2].set_title('Tank Diameter (px → m ÷ by 1.2)')
axes[2].legend()

plt.tight_layout()
plt.savefig('../results/eda_size_dist.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nDiameter stats:')
for pct in [10, 25, 50, 75, 90, 99]:
    print(f'  P{pct:2d}: {np.percentile(diameters, pct):.1f} px  '
          f'({np.percentile(diameters, pct)*1.2:.0f} m)')


## 3. Annotation Density per Image


In [ ]:
counts = [len(v) for v in annotations.values()]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(counts, bins=30, color='#ff4d6d', edgecolor='black', alpha=0.85)
ax.set_xlabel('Tanks per image'); ax.set_ylabel('Number of images')
ax.set_title('Annotation Density Distribution')
ax.axvline(np.mean(counts), color='yellow', lw=2, linestyle='--',
           label=f'Mean = {np.mean(counts):.0f}')
ax.legend()
plt.tight_layout()
plt.savefig('../results/eda_density.png', dpi=120, bbox_inches='tight')
plt.show()


## 4. Aspect Ratio Analysis (should be ~1.0 for circles)


In [ ]:
aspects = [w/max(h, 1) for w, h in zip(widths, heights)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(aspects, bins=50, range=(0, 3), color='#ff9f43', edgecolor='black', alpha=0.85)
ax.axvline(1.0, color='cyan', lw=2, linestyle='--', label='Ideal circle (AR=1.0)')
ax.set_xlabel('Aspect Ratio (W/H)'); ax.set_title('Bounding Box Aspect Ratio')
ax.legend()
print(f'Mean aspect ratio: {np.mean(aspects):.3f}  (expected ~1.0 for circular tanks)')
print(f'Std:               {np.std(aspects):.3f}')
plt.tight_layout(); plt.show()


## 5. Visualise Sample Images with Annotations


In [ ]:
# Show 4 random annotated images with GT boxes
annotated_imgs = [p for p in img_paths if p.stem in annotations]
sample = np.random.choice(annotated_imgs, min(4, len(annotated_imgs)), replace=False)

fig, axes = plt.subplots(1, len(sample), figsize=(5*len(sample), 5))
if len(sample) == 1: axes = [axes]

for ax, img_path in zip(axes, sample):
    img = np.array(Image.open(img_path).convert('RGB'))
    ax.imshow(img)
    for b in annotations.get(img_path.stem, []):
        rect = patches.Rectangle((b[0],b[1]), b[2]-b[0], b[3]-b[1],
                                  lw=1, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
    n = len(annotations.get(img_path.stem, []))
    ax.set_title(f'{img_path.stem[:20]}\n{n} tanks', fontsize=8)
    ax.axis('off')

plt.tight_layout(); plt.show()


## 6. Processed Tile Statistics


In [ ]:
for split in ['train', 'val', 'test']:
    lbl_dir = PROCESSED_DIR / 'labels' / split
    if not lbl_dir.exists(): continue
    files = list(lbl_dir.glob('*.txt'))
    total_boxes = sum(len(f.read_text().strip().splitlines()) for f in files)
    empty = sum(1 for f in files if not f.read_text().strip())
    print(f'{split:8s}: {len(files):5d} tiles | {total_boxes:6d} boxes | {empty:4d} empty tiles ({empty/max(len(files),1)*100:.0f}%)')


## 7. Sample Processed Tiles (post-tiling)


In [ ]:
train_imgs = sorted((PROCESSED_DIR / 'images' / 'train').glob('*.jpg'))
sample = np.random.choice(train_imgs, min(6, len(train_imgs)), replace=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flatten(), sample):
    img = np.array(Image.open(img_path).convert('RGB'))
    ax.imshow(img)
    lbl = PROCESSED_DIR / 'labels' / 'train' / (img_path.stem + '.txt')
    if lbl.exists():
        for line in lbl.read_text().strip().splitlines():
            p = list(map(float, line.split()))
            if len(p) == 5:
                cx,cy,bw,bh = p[1],p[2],p[3],p[4]
                x1=(cx-bw/2)*TILE_W; y1=(cy-bh/2)*TILE_H
                rect = patches.Rectangle((x1,y1),bw*TILE_W,bh*TILE_H,
                                          lw=1.5,edgecolor='lime',facecolor='none')
                ax.add_patch(rect)
    ax.set_title(img_path.stem[-30:], fontsize=7); ax.axis('off')

plt.tight_layout(); plt.show()
